# UdaPlay 01 — RAG Pipeline

**Author: Sam Sepassi**

This notebook prepares a persistent ChromaDB vector store from the local
`games/` JSON dataset so the UdaPlay agent (notebook 02) can answer
questions via Retrieval-Augmented Generation.

Pipeline:

1. Load every `games/NNN.json` record
2. Convert each record into a single embeddable document
3. Embed and persist into ChromaDB (`chromadb/` directory)
4. Sanity-check with three semantic queries


## 1. Setup


In [ ]:
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from lib.vector_store import VectorStoreManager, load_games_from_directory


## 2. Inspect the raw dataset


In [ ]:
GAMES_DIR = Path('games')
files = sorted(GAMES_DIR.glob('*.json'))
print(f'Found {len(files)} game JSON files')
with files[0].open() as fp:
    print(json.dumps(json.load(fp), indent=2))


## 3. Convert JSON records into embeddable documents

`load_games_from_directory` reads each file and produces a `GameDocument`
whose `text` is a single natural-language summary — that's what the
embedder sees, so the document text deliberately includes name, year,
platform, developer, publisher, genre, and description.


In [ ]:
documents = load_games_from_directory(GAMES_DIR)
print(f'Built {len(documents)} documents')
print('---')
print(documents[0].text)
print('---')
print(documents[0].metadata)


## 4. Build the persistent vector store

ChromaDB writes everything to the `chromadb/` directory so the agent
notebook can re-use the same index without re-embedding. We `reset()`
first so re-runs are idempotent.


In [ ]:
store = VectorStoreManager(persist_directory='chromadb')
store.reset()
written = store.add_games(documents)
print(f'Wrote {written} documents; collection now contains {store.count()} rows')


Peek at the first few persisted docs to confirm the round-trip:


In [ ]:
for row in list(store.peek(3)):
    print(row['id'], '->', row['metadata'].get('name'))


## 5. Semantic search smoke tests

Three queries that map onto the project specification's example
questions. We print the top hit and its distance.


In [ ]:
QUERIES = [
    'Who developed FIFA 21?',
    'When was God of War Ragnarok released?',
    'What platform was Pokemon Red launched on?',
]

for q in QUERIES:
    hits = store.query(q, k=3)
    print(f'Q: {q}')
    for h in hits:
        meta = h['metadata']
        print(f"  - {meta.get('name')} ({meta.get('year')}) [dist={h['distance']:.3f}]")
    print()


## 6. Done

The vector store is now persisted to `chromadb/`. Move on to
`Udaplay_02_solution_project.ipynb` to run the agent.
